# 03 — Sentiment Analysis (FinBERT)

Анализ тональности финансовых новостей с помощью **FinBERT** (ProsusAI/finbert):
- Загрузка новостей из `data/raw/{ticker}/news.json`
- Прогон через FinBERT (offline ETL)
- Агрегация sentiment по дням
- Сохранение в `data/processed/{ticker}/sentiment.parquet`

> Требует: `transformers`, `torch` (`uv add transformers torch`)

In [1]:
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "backend"))

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META"]
print(f"Root: {ROOT}")

Root: /Users/dvank1mang1/EquiSense


## 1. Загрузка FinBERT

In [2]:
from transformers import pipeline

print("Загружаем FinBERT...")
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=-1,
    truncation=True,
    max_length=512,
    top_k=None,
)
print("FinBERT загружен.")

Загружаем FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 54268.11it/s]


BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT загружен.


## 2. Вспомогательные функции

In [3]:
def score_texts(texts: list[str], batch_size: int = 16) -> list[dict]:
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        outputs = finbert(batch)
        for out in outputs:
            scores = {item["label"].lower(): item["score"] for item in out}
            net = scores.get("positive", 0) - scores.get("negative", 0)
            results.append({
                "positive": scores.get("positive", 0),
                "negative": scores.get("negative", 0),
                "neutral": scores.get("neutral", 0),
                "sentiment_score": net,
            })
    return results


def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    daily = df.groupby("date").agg(
        sentiment_score=("sentiment_score", "mean"),
        news_count=("sentiment_score", "count"),
        positive_ratio=("positive", "mean"),
        negative_ratio=("negative", "mean"),
    ).reset_index()
    daily["sentiment_momentum"] = daily["sentiment_score"].diff()
    return daily


def load_news(ticker: str) -> list[dict]:
    path = RAW_DIR / ticker / "news.json"
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)


print("Функции готовы.")

Функции готовы.


## 3. Прогон через FinBERT

In [4]:
sentiment_frames = {}

for ticker in TICKERS:
    news_items = load_news(ticker)
    if not news_items:
        print(f"[SKIP] {ticker}: нет данных новостей")
        continue

    print(f"[{ticker}] {len(news_items)} новостей...")

    records = []
    for item in news_items:
        title = item.get("title", "") or ""
        summary = item.get("summary", "") or ""
        text = (title + " " + summary).strip()[:512]
        if not text:
            continue
        ts = item.get("published") or item.get("datetime") or ""
        try:
            date = pd.to_datetime(ts).date()
        except Exception:
            continue
        records.append({"date": date, "text": text})

    if not records:
        print(f"  [SKIP] {ticker}: нет валидных записей")
        continue

    texts = [r["text"] for r in records]
    scores = score_texts(texts)

    df = pd.DataFrame(records)
    for key in ["sentiment_score", "positive", "negative", "neutral"]:
        df[key] = [s[key] for s in scores]

    daily = aggregate_daily(df)
    sentiment_frames[ticker] = daily

    out_dir = PROCESSED_DIR / ticker
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "sentiment.parquet"
    daily.to_parquet(out_path, index=False)
    print(f"  Сохранено: {len(daily)} дней → {out_path}")

print(f"\nГотово: {list(sentiment_frames.keys())}")

[SKIP] AAPL: нет данных новостей
[SKIP] MSFT: нет данных новостей
[SKIP] GOOGL: нет данных новостей
[SKIP] AMZN: нет данных новостей
[SKIP] TSLA: нет данных новостей
[SKIP] NVDA: нет данных новостей
[SKIP] META: нет данных новостей

Готово: []


## 4. Визуализация sentiment

In [5]:
import matplotlib.pyplot as plt

plt.style.use("dark_background")

if not sentiment_frames:
    print("Нет данных для визуализации — сначала скачайте новости (ноутбук 01).")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    for ticker, df in sentiment_frames.items():
        df_sorted = df.sort_values("date")
        axes[0].plot(pd.to_datetime(df_sorted["date"]),
                     df_sorted["sentiment_score"].rolling(5).mean(),
                     label=ticker, alpha=0.8)

    axes[0].axhline(0, color="white", linestyle="--", alpha=0.3)
    axes[0].set_title("Rolling 5d Sentiment Score по тикерам")
    axes[0].set_ylabel("score (positive - negative)")
    axes[0].legend(fontsize=8)

    for ticker, df in sentiment_frames.items():
        df_sorted = df.sort_values("date")
        axes[1].plot(pd.to_datetime(df_sorted["date"]),
                     df_sorted["news_count"].rolling(5).mean(),
                     label=ticker, alpha=0.8)

    axes[1].set_title("Rolling 5d News Count")
    axes[1].set_ylabel("кол-во новостей")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

Нет данных для визуализации — сначала скачайте новости (ноутбук 01).
